# Generated Data Analysis Notebook

Ten notebook sluzy do szczegolowego testowania danych wygenerowanych przez projekt Data Generator.

Mozesz pracowac w dwoch trybach:

- `generate` - generuje dane w pamieci na podstawie aktualnych ustawien projektu.
- `load` - laduje juz zapisany plik z katalogu `export/`, zgodnie z `saving_format` w `settings/main_settings.py`.

Domyslnie notebook nie zapisuje nowych plikow. Do testow najlepiej zaczac od mniejszej probki danych.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd


def find_project_root(start_path: Path) -> Path:
    for path in [start_path, *start_path.parents]:
        if (path / 'main.py').exists() and (path / 'source' / 'pl_postal_code.csv').exists():
            return path
    raise FileNotFoundError('Could not find project root with main.py and source/pl_postal_code.csv')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

try:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter, PercentFormatter
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print('matplotlib is not installed. Tables will work, charts will be skipped.')

try:
    from itables import init_notebook_mode, show
    init_notebook_mode(all_interactive=False)
    HAS_ITABLES = True
except ImportError:
    HAS_ITABLES = False
    print('itables is not installed. Static pandas display will be used.')


def show_table(data, **kwargs):
    if HAS_ITABLES:
        show(data, classes='display nowrap compact', scrollX=True, maxBytes=0, **kwargs)
    else:
        display(data)


CHART_FIGSIZE = (11, 5)
CHART_COLORS = {
    'primary': '#2F6F9F',
    'secondary': '#5B8E3E',
    'accent': '#C7782A',
    'neutral': '#6B7280',
}

if HAS_MATPLOTLIB:
    plt.rcParams.update({
        'figure.figsize': CHART_FIGSIZE,
        'axes.titlesize': 13,
        'axes.titleweight': 'bold',
        'axes.labelsize': 11,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'axes.edgecolor': '#D1D5DB',
        'axes.linewidth': 0.8,
        'grid.color': '#E5E7EB',
        'grid.linewidth': 0.8,
        'font.family': 'Calibri',
    })


def format_pln_axis(axis):
    axis.set_major_formatter(FuncFormatter(lambda value, _: f'{value / 1_000_000:.1f}M PLN' if abs(value) >= 1_000_000 else f'{value / 1_000:.0f}k PLN'))


def apply_chart_style(ax, title, xlabel, ylabel, grid_axis='y'):
    ax.set_title(title, pad=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis=grid_axis, alpha=0.9)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#D1D5DB')
    ax.spines['bottom'].set_color('#D1D5DB')
    return ax


def finish_chart():
    plt.tight_layout()
    plt.show()


def plot_horizontal_bar(series, title, xlabel, ylabel, color=None, percent=False, pln=False):
    ax = series.plot(kind='barh', figsize=CHART_FIGSIZE, color=color or CHART_COLORS['primary'])
    apply_chart_style(ax, title, xlabel, ylabel, grid_axis='x')
    if percent:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    if pln:
        format_pln_axis(ax.xaxis)
    finish_chart()
    return ax


def plot_vertical_bar(series, title, xlabel, ylabel, color=None, percent=False, pln=False, rotation=35):
    ax = series.plot(kind='bar', figsize=CHART_FIGSIZE, color=color or CHART_COLORS['secondary'])
    apply_chart_style(ax, title, xlabel, ylabel, grid_axis='y')
    if percent:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    if pln:
        format_pln_axis(ax.yaxis)
    plt.xticks(rotation=rotation, ha='right')
    finish_chart()
    return ax


def plot_line(series, title, xlabel, ylabel, color=None, pln=False, rotation=45):
    ax = series.plot(figsize=CHART_FIGSIZE, marker='o', linewidth=2.0, markersize=4, color=color or CHART_COLORS['secondary'])
    apply_chart_style(ax, title, xlabel, ylabel, grid_axis='y')
    if pln:
        format_pln_axis(ax.yaxis)
    plt.xticks(rotation=rotation, ha='right')
    finish_chart()
    return ax


def plot_histogram(series, title, xlabel, ylabel, bins=30, color=None, percent_x=False):
    ax = series.plot(kind='hist', bins=bins, figsize=CHART_FIGSIZE, color=color or CHART_COLORS['accent'], edgecolor='white')
    apply_chart_style(ax, title, xlabel, ylabel, grid_axis='y')
    if percent_x:
        ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    finish_chart()
    return ax


print(f'Project root: {PROJECT_ROOT}')
print(f'Current working directory: {Path.cwd()}')
print(f'Interactive tables: {HAS_ITABLES}')
print(f'Charts enabled: {HAS_MATPLOTLIB}')


## 2. Configuration

Ustaw `DATA_MODE`:

- `generate` - wygeneruje dane w pamieci.
- `load` - zaladuje dataset z katalogu `export/` w formacie ustawionym w `settings/main_settings.py`.

W trybie `generate` mozesz zmniejszyc wolumen danych, zeby szybko testowac logike.

In [ ]:
DATA_MODE = 'load'  # 'load' or 'generate'

# Used only when DATA_MODE = 'load'. The file extension is taken from settings/main_settings.py saving_format.
DATASET_NAME = 'sales_data'

# Used only when DATA_MODE = 'generate'.
TEST_RANDOM_STATE = 42
TEST_CUSTOMER_ROWS = 1_000
TEST_PRODUCT_ROWS = 3_000
TEST_ORDER_ROWS = 5_000

DATA_MODE, DATASET_NAME


## 3. Load Or Generate Data

In [ ]:
if 'PROJECT_ROOT' not in globals():
    from pathlib import Path
    import os
    import sys
    import pandas as pd

    def find_project_root(start_path: Path) -> Path:
        for path in [start_path, *start_path.parents]:
            if (path / 'main.py').exists() and (path / 'source' / 'pl_postal_code.csv').exists():
                return path
        raise FileNotFoundError('Could not find project root with main.py and source/pl_postal_code.csv')

    PROJECT_ROOT = find_project_root(Path.cwd().resolve())
    os.chdir(PROJECT_ROOT)
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

if 'show_table' not in globals():
    def show_table(data, **kwargs):
        display(data)


def get_configured_data_path(dataset_name: str) -> Path:
    import settings.main_settings as s

    supported_datasets = {'sales_data'}
    if dataset_name not in supported_datasets:
        raise ValueError(f'This notebook is designed for sales_data analysis. Unsupported dataset name: {dataset_name}')

    if s.saving_format not in {'csv', 'parquet'}:
        raise ValueError(f'Unsupported saving_format in settings/main_settings.py: {s.saving_format}')

    return PROJECT_ROOT / 'export' / f'{dataset_name}.{s.saving_format}'


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'File does not exist: {path}')
    if path.suffix == '.parquet':
        return pd.read_parquet(path)
    if path.suffix == '.csv':
        return pd.read_csv(path, sep=';')
    raise ValueError(f'Unsupported file extension: {path.suffix}')


def generate_sales_data() -> pd.DataFrame:
    import settings.main_settings as s
    import func.generator as g

    s.random_state = TEST_RANDOM_STATE
    s.customer_data_rows = TEST_CUSTOMER_ROWS
    s.product_data_rows = TEST_PRODUCT_ROWS
    s.order_data_rows = TEST_ORDER_ROWS
    s.generated_files = ['sales_data']

    return g.run_generation_pipeline()['sales_data']


def normalize_sales_schema(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    defaults = {
        'discount_rate': 0.0,
        'payment_provider': 'Unknown',
        'product_popularity': 1.0,
        'subcategory': 'Unknown',
        'product_type': 'Unknown',
        'product_series': 'Unknown',
        'product_model': 'Unknown',
        'product_name': 'Unknown',
    }
    for column, value in defaults.items():
        if column not in data.columns:
            data[column] = value

    numeric_columns = [
        'quantity', 'net_purchase_price', 'sales_margin', 'sales_price_net', 'sales_price_gross',
        'discount_rate', 'product_popularity'
    ]
    for column in numeric_columns:
        if column in data.columns:
            data[column] = pd.to_numeric(data[column], errors='coerce')

    if 'order_date' in data.columns:
        data['order_date'] = pd.to_datetime(data['order_date'], errors='coerce')

    if 'sales_price_gross_after_discount' not in data.columns:
        if {'sales_price_gross', 'discount_rate'}.issubset(data.columns):
            data['sales_price_gross_after_discount'] = data['sales_price_gross'] * (1 - data['discount_rate'])
        else:
            data['sales_price_gross_after_discount'] = np.nan

    if 'sales_price_net_after_discount' not in data.columns:
        if {'sales_price_net', 'discount_rate'}.issubset(data.columns):
            data['sales_price_net_after_discount'] = data['sales_price_net'] * (1 - data['discount_rate'])
        else:
            data['sales_price_net_after_discount'] = np.nan

    if 'revenue_gross' not in data.columns:
        if {'quantity', 'sales_price_gross_after_discount'}.issubset(data.columns):
            data['revenue_gross'] = data['quantity'] * data['sales_price_gross_after_discount']
        else:
            data['revenue_gross'] = np.nan

    if 'category' not in data.columns:
        data['category'] = 'Unknown'
    if 'subcategory' not in data.columns:
        data['subcategory'] = 'Unknown'
    if 'brand' not in data.columns:
        data['brand'] = 'Unknown'
    if 'product_name' in data.columns and (data['product_name'] == 'Unknown').all():
        data['product_name'] = data[['product_type', 'brand', 'product_series', 'product_model']].astype(str).agg(' '.join, axis=1)
    if 'city' not in data.columns:
        data['city'] = 'Unknown'
    if 'gender' not in data.columns:
        data['gender'] = 'Unknown'
    if 'age' not in data.columns:
        data['age'] = np.nan

    return data


if DATA_MODE == 'generate':
    sales = generate_sales_data()
elif DATA_MODE == 'load':
    DATA_PATH = get_configured_data_path(DATASET_NAME)
    print(f'Loading dataset from: {DATA_PATH}')
    sales = load_dataset(DATA_PATH)
else:
    raise ValueError('DATA_MODE must be generate or load')

sales = normalize_sales_schema(sales)
show_table(sales.head(20))


## 4. Basic Shape And Schema

In [ ]:
print(f'Rows: {len(sales):,}')
print(f'Columns: {sales.shape[1]:,}')
schema_table = pd.DataFrame({
    'column': sales.columns,
    'dtype': [sales[col].dtype for col in sales.columns],
    'missing': [sales[col].isna().sum() for col in sales.columns],
    'missing_pct': [sales[col].isna().mean() for col in sales.columns],
})
show_table(schema_table)

## 5. Data Quality Checks

In [ ]:
required_columns = [
    'order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
    'category', 'brand', 'net_purchase_price', 'sales_margin',
    'sales_price_net', 'sales_price_gross', 'discount_rate', 'revenue_gross',
    'sales_price_gross_after_discount', 'sales_price_net_after_discount'
]

checks = []
checks.append(('required_columns_present', set(required_columns).issubset(sales.columns)))
checks.append(('no_negative_quantity', sales['quantity'].dropna().gt(0).all()))
checks.append(('no_negative_prices', sales[['net_purchase_price', 'sales_price_net', 'sales_price_gross']].dropna().ge(0).all().all()))
checks.append(('discount_rate_between_0_and_1', sales['discount_rate'].dropna().between(0, 1).all()))
checks.append(('revenue_non_negative', sales['revenue_gross'].dropna().ge(0).all()))
checks.append(('order_date_not_null', sales['order_date'].notna().all()))
if 'unit_price_gross' in sales.columns:
    expected_revenue = sales['quantity'] * sales['unit_price_gross'] * (1 - sales['discount_rate'])
    checks.append(('revenue_matches_line_price', np.allclose(sales['revenue_gross'], expected_revenue, equal_nan=True)))
checks.append(('no_missing_core_ids', sales[['order_id', 'customer_id', 'product_id']].notna().all().all()))

quality = pd.DataFrame(checks, columns=['check', 'passed'])
show_table(quality)

if not quality['passed'].all():
    print('Some checks failed. Inspect failed checks before using the data.')


## 6. Revenue And Order Summary

In [ ]:
order_revenue = sales.groupby('order_id')['revenue_gross'].sum()

summary = pd.Series({
    'rows': len(sales),
    'orders': sales['order_id'].nunique(),
    'customers': sales['customer_id'].nunique(),
    'products_sold': sales['product_id'].nunique(),
    'date_min': sales['order_date'].min(),
    'date_max': sales['order_date'].max(),
    'revenue_gross': sales['revenue_gross'].sum(),
    'avg_order_value': order_revenue.mean(),
    'median_order_value': order_revenue.median(),
    'avg_line_value': sales['revenue_gross'].mean(),
    'avg_quantity_per_line': sales['quantity'].mean(),
    'discounted_lines_pct': (sales['discount_rate'] > 0).mean(),
    'avg_discount_rate': sales['discount_rate'].mean(),
})

show_table(summary.to_frame('value'))

## 7. Category Analysis

In [ ]:
category = sales.groupby('category').agg(
    rows=('order_id', 'size'),
    orders=('order_id', 'nunique'),
    revenue=('revenue_gross', 'sum'),
    quantity=('quantity', 'sum'),
    products=('product_id', 'nunique'),
    avg_margin=('sales_margin', 'mean'),
    avg_discount=('discount_rate', 'mean'),
)
category['revenue_share'] = category['revenue'] / category['revenue'].sum()
category['row_share'] = category['rows'] / category['rows'].sum()
category = category.sort_values('revenue', ascending=False)
show_table(category)

if HAS_MATPLOTLIB:
    plot_data = category['revenue_share'].head(12).sort_values()
    plot_horizontal_bar(
        plot_data,
        title='Revenue Share by Category',
        xlabel='Share of gross revenue',
        ylabel='Category',
        color=CHART_COLORS['primary'],
        percent=True,
    )


## 8. Margin Mass Analysis

In [ ]:
sales['margin_mass_net'] = (sales['sales_price_net_after_discount'] - sales['net_purchase_price']) * sales['quantity']

margin_by_category = sales.groupby('category').agg(
    margin_mass=('margin_mass_net', 'sum'),
    revenue=('revenue_gross', 'sum'),
    avg_margin=('sales_margin', 'mean'),
)
margin_by_category['margin_mass_share'] = margin_by_category['margin_mass'] / margin_by_category['margin_mass'].sum()
margin_by_category = margin_by_category.sort_values('margin_mass', ascending=False)
show_table(margin_by_category)

if HAS_MATPLOTLIB:
    plot_data = margin_by_category['margin_mass_share'].head(12).sort_values()
    plot_horizontal_bar(
        plot_data,
        title='Margin Mass Share by Category',
        xlabel='Share of net margin mass',
        ylabel='Category',
        color=CHART_COLORS['secondary'],
        percent=True,
    )


## 9. Brand Analysis

In [ ]:
brand = sales.groupby(['category', 'brand']).agg(
    rows=('order_id', 'size'),
    revenue=('revenue_gross', 'sum'),
    quantity=('quantity', 'sum'),
    products=('product_id', 'nunique'),
).reset_index()
brand['category_revenue_share'] = brand['revenue'] / brand.groupby('category')['revenue'].transform('sum')

top_brand_per_category = brand.sort_values(['category', 'revenue'], ascending=[True, False]).groupby('category').head(5)
show_table(top_brand_per_category)

overall_top_brands = brand.groupby('brand').agg(revenue=('revenue', 'sum'), rows=('rows', 'sum')).sort_values('revenue', ascending=False).head(20)
show_table(overall_top_brands)

## 10. Product Popularity And Long Tail

In [ ]:
product = sales.groupby(['product_id', 'product_name', 'category', 'subcategory', 'product_type', 'brand']).agg(
    rows=('order_id', 'size'),
    revenue=('revenue_gross', 'sum'),
    quantity=('quantity', 'sum'),
    product_popularity=('product_popularity', 'mean'),
).sort_values('revenue', ascending=False)

show_table(product.head(30))

product_revenue_share = product['revenue'] / product['revenue'].sum()
print('Top 1% products revenue share:', product_revenue_share.head(max(1, int(len(product) * 0.01))).sum())
print('Top 10% products revenue share:', product_revenue_share.head(max(1, int(len(product) * 0.10))).sum())

if HAS_MATPLOTLIB:
    plot_line(
        product['revenue'].reset_index(drop=True).head(100),
        title='Revenue Concentration in Top 100 Products',
        xlabel='Product rank by gross revenue',
        ylabel='Gross revenue',
        pln=True,
        rotation=0,
    )


## 11. TOP20 Best-Selling Products


In [ ]:
top20_products = sales.groupby(['product_id', 'product_name', 'category', 'subcategory', 'product_type', 'brand']).agg(
    orders=('order_id', 'nunique'),
    rows=('order_id', 'size'),
    quantity=('quantity', 'sum'),
    revenue=('revenue_gross', 'sum'),
    avg_line_value=('revenue_gross', 'mean'),
    avg_discount=('discount_rate', 'mean'),
    avg_margin=('sales_margin', 'mean'),
).reset_index()

top20_products['revenue_share'] = top20_products['revenue'] / top20_products['revenue'].sum()
top20_products['quantity_share'] = top20_products['quantity'] / top20_products['quantity'].sum()

top20_products_by_revenue = top20_products.sort_values('revenue', ascending=False).head(20)
top20_products_by_quantity = top20_products.sort_values('quantity', ascending=False).head(20)
top20_products_by_orders = top20_products.sort_values('orders', ascending=False).head(20)

show_table(top20_products_by_revenue)
show_table(top20_products_by_quantity)
show_table(top20_products_by_orders)


## 12. Seasonality

In [ ]:
sales['year_month'] = sales['order_date'].dt.to_period('M').astype(str)
sales['month'] = sales['order_date'].dt.month
sales['weekday'] = sales['order_date'].dt.day_name()

monthly = sales.groupby('year_month').agg(
    rows=('order_id', 'size'),
    orders=('order_id', 'nunique'),
    revenue=('revenue_gross', 'sum'),
)
show_table(monthly)

weekday = sales.groupby('weekday').agg(
    rows=('order_id', 'size'),
    revenue=('revenue_gross', 'sum'),
).reindex(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'])
show_table(weekday)

if HAS_MATPLOTLIB:
    plot_line(
        monthly['revenue'],
        title='Monthly Gross Revenue',
        xlabel='Year-month',
        ylabel='Gross revenue',
        pln=True,
        rotation=45,
    )
    plot_vertical_bar(
        weekday['revenue'],
        title='Gross Revenue by Weekday',
        xlabel='Weekday',
        ylabel='Gross revenue',
        color=CHART_COLORS['secondary'],
        pln=True,
        rotation=35,
    )


## 13. Discounts

In [ ]:
discount = sales.groupby('category').agg(
    rows=('order_id', 'size'),
    discounted_lines=('discount_rate', lambda x: (x > 0).sum()),
    avg_discount=('discount_rate', 'mean'),
    max_discount=('discount_rate', 'max'),
    revenue=('revenue_gross', 'sum'),
)
discount['discounted_lines_pct'] = discount['discounted_lines'] / discount['rows']
show_table(discount.sort_values('discounted_lines_pct', ascending=False))

if HAS_MATPLOTLIB:
    discounted = sales.loc[sales['discount_rate'] > 0, 'discount_rate']
    if len(discounted) > 0:
        plot_histogram(
            discounted,
            title='Discount Rate Distribution',
            xlabel='Discount rate',
            ylabel='Number of order lines',
            bins=30,
            color=CHART_COLORS['accent'],
            percent_x=True,
        )
    else:
        print('No discounted lines available for histogram.')


## 14. Payments And Delivery

In [ ]:
for column in ['payment_method', 'payment_provider', 'delivery_method']:
    table = sales.groupby(column).agg(
        rows=('order_id', 'size'),
        orders=('order_id', 'nunique'),
        revenue=('revenue_gross', 'sum'),
    ).sort_values('rows', ascending=False)
    table['row_share'] = table['rows'] / table['rows'].sum()
    print(column)
    show_table(table)

if HAS_MATPLOTLIB:
    plot_horizontal_bar(
        sales['payment_method'].value_counts(normalize=True).sort_values(),
        title='Payment Method Share',
        xlabel='Share of order lines',
        ylabel='Payment method',
        color=CHART_COLORS['primary'],
        percent=True,
    )
    plot_horizontal_bar(
        sales['delivery_method'].value_counts(normalize=True).sort_values(),
        title='Delivery Method Share',
        xlabel='Share of order lines',
        ylabel='Delivery method',
        color=CHART_COLORS['secondary'],
        percent=True,
    )


## 15. Customers

In [ ]:
customer = sales.groupby('customer_id').agg(
    orders=('order_id', 'nunique'),
    rows=('order_id', 'size'),
    revenue=('revenue_gross', 'sum'),
    age=('age', 'first'),
    gender=('gender', 'first'),
    city=('city', 'first'),
).sort_values('revenue', ascending=False)

show_table(customer.head(30))
show_table(customer[['orders', 'rows', 'revenue', 'age']].describe())

city = customer.groupby('city').agg(customers=('orders', 'size'), revenue=('revenue', 'sum')).sort_values('revenue', ascending=False).head(20)
show_table(city)

## 16. Export Diagnostic Tables

Opcjonalnie mozesz zapisac wybrane tabele diagnostyczne do katalogu `analysis_output/`.

In [ ]:
SAVE_DIAGNOSTICS = False
OUTPUT_DIR = PROJECT_ROOT / 'analysis_output'

if SAVE_DIAGNOSTICS:
    OUTPUT_DIR.mkdir(exist_ok=True)
    if 'category' in globals():
        category.to_csv(OUTPUT_DIR / 'category_summary.csv')
    if 'margin_by_category' in globals():
        margin_by_category.to_csv(OUTPUT_DIR / 'margin_by_category.csv')
    if 'top_brand_per_category' in globals():
        top_brand_per_category.to_csv(OUTPUT_DIR / 'top_brand_per_category.csv', index=False)
    if 'product' in globals():
        product.head(500).to_csv(OUTPUT_DIR / 'top_products.csv')
    if 'monthly' in globals():
        monthly.to_csv(OUTPUT_DIR / 'monthly_summary.csv')
    print(f'Diagnostics saved to: {OUTPUT_DIR}')
else:
    print('Diagnostics were not saved. Set SAVE_DIAGNOSTICS = True to save them.')


## 17. Detailed Interactive Table

Ta sekcja pozwala przejrzec szczegolowe rekordy `sales_data` w interaktywnej tabeli. Mozesz ograniczyc liczbe wierszy, wybrac kolumny oraz ustawic proste filtry.


In [ ]:
DETAIL_LIMIT = 1_000
DETAIL_COLUMNS = [
    'order_id', 'order_date', 'customer_id', 'product_id', 'product_name',
    'category', 'subcategory', 'product_type', 'brand', 'product_series', 'product_model',
    'quantity', 'sales_price_gross', 'discount_rate', 'revenue_gross',
    'payment_method', 'payment_provider', 'delivery_method', 'city'
]

# Optional filters. Use None to disable a filter.
DETAIL_CATEGORY = None          # example: 'Clothing'
DETAIL_BRAND = None             # example: 'Nike'
DETAIL_DATE_FROM = None         # example: '2024-01-01'
DETAIL_DATE_TO = None           # example: '2024-12-31'
DETAIL_MIN_REVENUE = None       # example: 100.0


def build_detail_table(data: pd.DataFrame) -> pd.DataFrame:
    detail = data.copy()

    if DETAIL_CATEGORY is not None:
        detail = detail[detail['category'] == DETAIL_CATEGORY]
    if DETAIL_BRAND is not None:
        detail = detail[detail['brand'] == DETAIL_BRAND]
    if DETAIL_DATE_FROM is not None:
        detail = detail[detail['order_date'] >= pd.to_datetime(DETAIL_DATE_FROM)]
    if DETAIL_DATE_TO is not None:
        detail = detail[detail['order_date'] <= pd.to_datetime(DETAIL_DATE_TO)]
    if DETAIL_MIN_REVENUE is not None:
        detail = detail[detail['revenue_gross'] >= DETAIL_MIN_REVENUE]

    selected_columns = [column for column in DETAIL_COLUMNS if column in detail.columns]
    detail = detail[selected_columns].sort_values(['order_date', 'order_id'], ascending=[False, False])

    return detail.head(DETAIL_LIMIT)


detail_table = build_detail_table(sales)
print(f'Detail rows displayed: {len(detail_table):,}')
show_table(detail_table)


## 18. Advanced Tables

Ta sekcja sprawdza tabele i zachowania dodane w wersji z zaawansowanymi danymi: kohorty i segmenty klientów, koszyk,
zwroty, ruch i kampanie, eksperymenty A/B, anomalie i brudne dane. W trybie `load` czyta pliki z `export/`
(uruchom wcześniej `python main.py`), w trybie `generate` generuje wszystkie tabele w pamięci.

In [ ]:
TABLE_NAMES = [
    'customer_data', 'order_data', 'product_data', 'return_data', 'session_data', 'campaign_data',
    'experiment_data', 'experiment_assignment_data', 'anomaly_data', 'quality_issue_data',
]


def get_tables() -> dict:
    import settings.main_settings as s
    import func.generator as g

    if DATA_MODE == 'generate':
        s.generated_files = list(g.DATASET_NAMES)
        return g.run_generation_pipeline()
    return {name: load_dataset(PROJECT_ROOT / 'export' / f'{name}.{s.saving_format}') for name in TABLE_NAMES}


tables = get_tables()
customers = tables['customer_data']
pd.Series({name: len(tables[name]) for name in TABLE_NAMES}, name='rows').to_frame()

### 18.1 Cohort retention

Udział klientów z kohorty (miesiąc pierwszego zakupu) kupujących ponownie po k miesiącach. Retencja powinna spadać po
pierwszym miesiącu. Klienci zarejestrowani przed okresem danych są pominięci (ich pierwsze zakupy są wcześniej).

In [ ]:
new_customers = customers.loc[pd.to_datetime(customers['signup_date']) >= sales['order_date'].min(), 'customer_id']
orders = sales[sales['customer_id'].isin(new_customers)].drop_duplicates('order_id')[['customer_id', 'order_date']]
month = orders['order_date'].dt.to_period('M')
first_month = month.groupby(orders['customer_id']).transform('min')
offset = (month - first_month).apply(lambda delta: delta.n)
active = pd.DataFrame({'customer_id': orders['customer_id'], 'cohort': first_month, 'offset': offset}).drop_duplicates()
cohort_size = active[active['offset'] == 0].groupby('cohort').size()
retention = active.groupby(['cohort', 'offset']).size().unstack(fill_value=0).div(cohort_size, axis=0)
show_table(retention.iloc[:12, :13].round(3))

### 18.2 Hidden segments and price tiers

`customer_segment`, `preferred_category` i `price_tier` to ukryta prawda do sprawdzania segmentacji i rekomendacji.

In [ ]:
per_customer = sales.groupby('customer_id').agg(orders=('order_id', 'nunique'), revenue=('revenue_gross', 'sum'))
per_customer = customers.set_index('customer_id')[['customer_segment', 'price_tier']].join(per_customer).fillna(0)
show_table(per_customer.groupby('customer_segment')[['orders', 'revenue']].mean().round(1))
show_table(sales.groupby('price_tier')['unit_price_gross'].median().round(0).to_frame('median_unit_price'))

### 18.3 Market basket: lift of product type pairs

In [ ]:
baskets = sales.groupby('order_id')['product_type'].agg(lambda types: sorted(set(types)))
pairs = baskets[baskets.map(len) > 1].explode().reset_index().merge(
    baskets[baskets.map(len) > 1].explode().reset_index(), on='order_id')
pairs = pairs[pairs['product_type_x'] < pairs['product_type_y']]
together = pairs.groupby(['product_type_x', 'product_type_y']).size().rename('orders')
type_share = baskets.explode().value_counts() / len(baskets)
lift = together / len(baskets) / (
    type_share.reindex(together.index.get_level_values(0)).to_numpy()
    * type_share.reindex(together.index.get_level_values(1)).to_numpy())
show_table(pd.DataFrame({'orders': together, 'lift': lift.round(1)}).query('orders >= 20')
           .sort_values('lift', ascending=False).head(15))

### 18.4 Returns by category

In [ ]:
returns = tables['return_data'].merge(tables['product_data'][['product_id', 'category']], on='product_id')
return_rate = (returns.groupby('category').size() / sales.groupby('category').size()).sort_values(ascending=False)
show_table(return_rate.round(3).to_frame('return_rate'))
show_table(returns.groupby(['category', 'return_reason']).size().unstack(fill_value=0).loc[return_rate.index[:2]])

### 18.5 Traffic, conversion and campaigns

In [ ]:
sessions = tables['session_data']
funnel = sessions.groupby('channel').agg(
    sessions=('session_id', 'size'),
    carts=('added_to_cart', 'sum'),
    checkouts=('reached_checkout', 'sum'),
    orders=('order_id', 'count'),
)
funnel['conversion'] = funnel['orders'] / funnel['sessions']
show_table(funnel.sort_values('conversion', ascending=False).round(3))

campaigns = tables['campaign_data']
attributed = sales.groupby('campaign_id')['revenue_gross'].sum()
campaigns = campaigns.assign(revenue=campaigns['campaign_id'].map(attributed).fillna(0))
roas = campaigns.groupby('channel')[['total_cost', 'revenue']].sum()
roas['roas'] = roas['revenue'] / roas['total_cost']
show_table(roas.round(1))

### 18.6 A/B experiments

Wynik każdego eksperymentu z 95% przedziałem ufności i prawdziwym efektem (`true_effect`) do porównania. Test
podziału (chi-kwadrat) powinien wykryć wpadkę w eksperymencie z `has_sample_ratio_mismatch = True`.

In [ ]:
import math

assignments = tables['experiment_assignment_data']
session_orders = sessions.set_index('session_id')['order_id']
rows = []
for experiment in tables['experiment_data'].itertuples():
    units = assignments[assignments['experiment_id'] == experiment.experiment_id]
    count_a, count_b = (units['variant'] == 'A').sum(), (units['variant'] == 'B').sum()
    expected_b = experiment.planned_split_b * (count_a + count_b)
    expected_a = (count_a + count_b) - expected_b
    chi2 = (count_a - expected_a) ** 2 / expected_a + (count_b - expected_b) ** 2 / expected_b
    row = {'experiment': experiment.experiment_name, 'metric': experiment.metric, 'true_effect': experiment.true_effect,
           'share_b': count_b / (count_a + count_b), 'srm_p_value': math.erfc(math.sqrt(chi2 / 2))}
    if experiment.metric == 'conversion_rate':
        converted = session_orders.reindex(units['unit_id']).notna().to_numpy()
        is_b = units['variant'].to_numpy() == 'B'
        pa, pb = converted[~is_b].mean(), converted[is_b].mean()
        se = math.sqrt((1 - pa) / (pa * (~is_b).sum()) + (1 - pb) / (pb * is_b.sum()))
        row.update(estimate=pb / pa - 1, ci_low=math.exp(math.log(pb / pa) - 1.96 * se) - 1,
                   ci_high=math.exp(math.log(pb / pa) + 1.96 * se) - 1)
    rows.append(row)
show_table(pd.DataFrame(rows).round(4))

### 18.7 Anomalies and data quality issues

`anomaly_data` i `is_fraud` to prawda do wykrywania anomalii; `quality_issue_data` (pusta przy
`data_quality_level = "none"`) do oceny czyszczenia danych.

In [ ]:
show_table(tables['anomaly_data'].groupby('anomaly_type').agg(events=('anomaly_id', 'size'),
                                                              rows=('affected_rows', 'sum')))
fraud_orders = sales.groupby('order_id').agg(value=('revenue_gross', 'sum'), fraud=('is_fraud', 'first'))
show_table(fraud_orders.groupby('fraud')['value'].describe().round(0))
issues = tables['quality_issue_data']
show_table(issues.groupby(['table_name', 'column_name', 'issue_type']).size().rename('issues').to_frame()
           if len(issues) else pd.DataFrame({'issues': [0]}))